#### Data Quality


**Criando Schemas e Volume Tables para Armazenamento de Dados em DeltaTable**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS costumers_registrations.silver
COMMENT "Schema com Data Quality Aplicado";

CREATE VOLUME IF NOT EXISTS costumers_registrations.silver.registrations
COMMENT "Volume com Data Quality Aplicado para Cadastros";


#### Imports

In [0]:
from pyspark.sql.types import *
from pyspark.testing.utils import assertSchemaEqual
from pyspark.sql.functions import *


**🔒 Schema Enforcement - Validação de schema**

```
Comparação de Schemas esperados entre: Schema Atual X Schema Esperado
```

In [0]:
# Schema esperado
schema_expected = StructType([
    StructField('id', StringType(), True),
    StructField('nome', StringType(), True),
    StructField('data_nascimento', DateType(), True),
    StructField('cpf', StringType(),True),
    StructField('cep', StringType(), True),
    StructField('cidade', StringType(), True),
    StructField('estado', StringType(), True),
    StructField('pais', StringType(), True),
    StructField('genero', StringType(), True),
    StructField('telefone', StringType(), True),
    StructField('email', StringType(), True),
    StructField('data_cadastro', DateType(), True)
])

df_registrations = spark.read.parquet('/Volumes/costumers_registrations/raw/registrations')

actual_schema = df_registrations.schema

try:
    """
    Em vez de deixar quebrar seco, abrimos oportunidade de Logar em MLflow, Registrar em tabela de auditoria, acionar alerta.
    """
    assertSchemaEqual(actual_schema, schema_expected) # Valida se os dois schemas são exatamente iguais (case sensitive)
    print("Schema válido")
except AssertionError as e:
    print("Schema inválido\n")
    print(e)

print('TESTE DE SCHEMA')
print(f"Schema Esperado:\n {schema_expected}\n")
print(f"Schema Atual:\n {actual_schema}\n")
print("\nTeste de Schema Concluído\n")

**Campos obrigatórios não nulos**

```
Campos obrigatóriamente não nulos validados
```

In [0]:


df_nulls = df_registrations.filter(
    col("id").isNull() |
    col("cpf").isNull() |
    col("email").isNull() |
    col("data_cadastro").isNull() |
    col("data_nascimento").isNull()
)

df_is_null = df_nulls.count()

if df_is_null == 0:
    print("Não há registros inválidos por nulos")
else:
    print(f"\nQuantidade de registros inválidos por nulos: {df_is_null}\n")
